# 08 – Temperature–Price Coupling Estimation

Estimates the coupling parameter **Γ** that links the temperature Lévy driver
dL^Y to the log-price state equation in the joint CARMA model:

$$dZ^X = A_X Z^X\,dt + B_X\,dL^X + \Gamma\,dL^Y$$

### Method
1. Align dL^Y (temperature) and dL^X (log-price) on their common hourly grid.
2. Compute the empirical cross-covariance function (CCF) at lags −72…+72 h.
3. Estimate the scalar coupling strength γ₀ = Cov(dL^X, dL^Y) / Var(dL^Y) by OLS.
4. In the state-space interpretation, **Γ = γ₀ · B_X** (same loading structure);
   this means the price CARMA is driven by dL^X + γ₀ dL^Y.

Inputs:  `data/increments/temp_inc.csv`, `data/increments/price_inc.csv`,  
         `results/carma_price_mle.json`  
Outputs: `results/coupling_params.json`, `figures/coupling_ccf.png`

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

from config import RES_DIR, FIG_DIR, INCR_DIR, DT_YEARS
from carma_utils import build_companion, load_params

# ── Load increments ───────────────────────────────────────────────────────────
dL_temp  = pd.read_csv(INCR_DIR / 'temp_inc.csv')['dL_temp'].to_numpy()
dL_price = pd.read_csv(INCR_DIR / 'price_inc.csv')['dL_price'].to_numpy()

print(f'Temperature increments : {len(dL_temp):,}')
print(f'Log-price increments   : {len(dL_price):,}')

## 1.  Align on common time grid

Temperature covers 2020–2024; log-price covers 2023–2024 (training window).
We align by taking the **tail** of the temperature series equal in length to the
price series, assuming both end at the same calendar date.

In [ ]:
n = min(len(dL_temp), len(dL_price))
dLY = dL_temp[-n:].copy()   # temperature (driver)
dLX = dL_price[-n:].copy()  # log-price   (driven)

print(f'Aligned length: {n:,} hourly observations')
print(f'\nBasic statistics on aligned series:')
for label, dL in [('dL^Y (temp)', dLY), ('dL^X (price)', dLX)]:
    print(f'  {label}: mean={dL.mean():.6f}  std={dL.std():.6f}  '
          f'skew={scipy_stats.skew(dL):.3f}  kurt={scipy_stats.kurtosis(dL, fisher=True):.3f}')

## 2.  Cross-covariance function

In [ ]:
max_lag = 72
lags = np.arange(-max_lag, max_lag + 1)

# Standardise for CCF
dLY_s = (dLY - dLY.mean()) / dLY.std()
dLX_s = (dLX - dLX.mean()) / dLX.std()

ccf = np.array([
    np.corrcoef(dLY_s[:n - abs(k)], dLX_s[max(0, k):n - max(0, -k)])[0, 1]
    for k in lags
])

ci = 1.96 / np.sqrt(n)
lag0_corr = float(np.corrcoef(dLY_s, dLX_s)[0, 1])

print(f'Cross-correlation at lag 0: {lag0_corr:.4f}')
print(f'95% confidence band: ±{ci:.4f}')
if abs(lag0_corr) > ci:
    print('=> Statistically significant contemporaneous coupling (Γ ≠ 0)')
else:
    print('=> No significant coupling at lag 0')

# Find significant lags
sig_lags = lags[np.abs(ccf) > ci]
if len(sig_lags) > 0:
    print(f'Significant lags (|CCF| > {ci:.4f}): {sig_lags.tolist()}')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(lags, ccf, width=0.8, color='steelblue', alpha=0.8)
ax.axhline( ci, color='r', ls='--', lw=1.2, label=f'±1.96/√n = ±{ci:.4f}')
ax.axhline(-ci, color='r', ls='--', lw=1.2)
ax.axvline(0, color='k', lw=0.8, ls=':')
ax.set_xlabel('Lag k (hours, temperature leads →)')
ax.set_ylabel('Cross-correlation')
ax.set_title('CCF: dL^Y (temperature) vs dL^X (log-price)')
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'coupling_ccf.png', dpi=150)
plt.show()

## 3.  OLS estimate of contemporaneous coupling γ₀

Model: $dL^X_t = \gamma_0\,dL^Y_t + \varepsilon_t$

OLS estimator: $\hat\gamma_0 = \text{Cov}(dL^X,\,dL^Y) / \text{Var}(dL^Y)$

Interpretation: after removing the shared temperature component $\hat\gamma_0 dL^Y_t$,
the residual $\varepsilon_t$ is the idiosyncratic log-price Lévy increment.

The state-space coupling vector is **Γ = γ₀ · B_X** where
$B_X = \sigma_X \, e_{p_X}$ is the noise loading for the standalone price CARMA.

In [ ]:
# OLS: regress dLX on dLY (no intercept; increments are mean-zero)
gamma0 = float(np.dot(dLY, dLX) / np.dot(dLY, dLY))

# Residual (idiosyncratic price noise)
eps = dLX - gamma0 * dLY

# R² and significance
ss_res = float(np.dot(eps, eps))
ss_tot = float(np.dot(dLX - dLX.mean(), dLX - dLX.mean()))
r2 = 1.0 - ss_res / ss_tot

# OLS standard error for gamma0
sigma_eps = np.sqrt(ss_res / (n - 1))
se_gamma0 = float(sigma_eps / np.sqrt(np.dot(dLY, dLY)))
t_stat    = gamma0 / se_gamma0
p_val     = 2 * float(scipy_stats.t.sf(abs(t_stat), df=n - 1))

print(f'Coupling estimate:')
print(f'  γ₀       = {gamma0:.6f}')
print(f'  SE(γ₀)   = {se_gamma0:.6f}')
print(f'  t-stat   = {t_stat:.3f}')
print(f'  p-value  = {p_val:.4e}')
print(f'  R²       = {r2:.6f}')
print()
print(f'Residual ε_t (idiosyncratic price noise):')
print(f'  mean={eps.mean():.6f}  std={eps.std():.6f}')
print(f'  Pearson corr(ε, dLY): {float(np.corrcoef(eps, dLY)[0,1]):.6f}  (should be ≈ 0)')

## 4.  State-space coupling vector Γ

In the CARMA state equation the coupling enters as:
$$dZ^X = A_X Z^X\,dt + B_X\,dL^X + \Gamma\,dL^Y$$

With the OLS estimate, we model $\Gamma = \gamma_0 B_X$, so the price CARMA
is effectively driven by $dL^X + \gamma_0 dL^Y$. This preserves the structure
assumed in the paper's closed-form characteristic function.

In [ ]:
params_price = load_params(RES_DIR / 'carma_price_mle.json')
p_p   = len(params_price['a_coeffs'])
sigma_price = float(params_price['sigma'])

B_X = np.zeros(p_p)
B_X[-1] = sigma_price

Gamma_vec = gamma0 * B_X   # (p_p,) vector

print(f'Price CARMA order p_X = {p_p}')
print(f'σ_X = {sigma_price:.6f}  yr^(-1/2)')
print(f'B_X = {np.round(B_X, 6)}')
print(f'γ₀  = {gamma0:.6f}')
print(f'Γ   = γ₀ · B_X = {np.round(Gamma_vec, 6)}')

## 5.  Diagnostic: residual distribution and CCF check

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Coupling residuals ε_t = dL^X − γ₀ dL^Y', fontsize=11)

# Residual histogram
ax = axes[0]
eps_s = (eps - eps.mean()) / eps.std()
xg = np.linspace(-6, 6, 300)
ax.hist(eps_s, bins=200, density=True, color='lightgray', edgecolor='none')
ax.plot(xg, scipy_stats.norm.pdf(xg), 'r-', lw=2, label='N(0,1)')
ax.set_xlim(-8, 8)
ax.set_title('Standardised residuals histogram')
ax.legend()

# Cross-correlation of residuals with dLY (should be zero)
ax = axes[1]
eps_s2 = (eps - eps.mean()) / eps.std()
ccf_eps = np.array([
    np.corrcoef(dLY_s[:n - abs(k)], eps_s2[max(0, k):n - max(0, -k)])[0, 1]
    for k in lags
])
ax.bar(lags, ccf_eps, width=0.8, color='steelblue', alpha=0.8)
ax.axhline( ci, color='r', ls='--', lw=1)
ax.axhline(-ci, color='r', ls='--', lw=1)
ax.set_xlabel('Lag (hours)')
ax.set_ylabel('Cross-correlation')
ax.set_title('CCF(dL^Y, ε) – should be zero after coupling')

# QQ of residuals
ax = axes[2]
(osm, osr), (slope, intercept, _) = scipy_stats.probplot(eps_s, dist='norm')
ax.scatter(osm, osr, s=1, alpha=0.3)
xl = np.array([osm.min(), osm.max()])
ax.plot(xl, slope*xl + intercept, 'r-', lw=2)
ax.set_xlabel('Theoretical N(0,1) quantiles')
ax.set_ylabel('Sample quantiles')
ax.set_title('QQ-plot: coupling residuals vs Normal')

plt.tight_layout()
plt.savefig(FIG_DIR / 'coupling_residuals.png', dpi=150)
plt.show()

# Ljung-Box on residuals
lb = acorr_ljungbox(eps, lags=[24, 48, 72], return_df=True)
print('Ljung-Box on coupling residuals:')
print(f'  lag24 p={lb["lb_pvalue"].iloc[0]:.4f}  '
      f'lag48 p={lb["lb_pvalue"].iloc[1]:.4f}  '
      f'lag72 p={lb["lb_pvalue"].iloc[2]:.4f}')

## 6.  Save coupling parameters

In [ ]:
coupling_params = {
    'gamma0':          gamma0,
    'se_gamma0':       se_gamma0,
    't_stat':          t_stat,
    'p_value':         p_val,
    'r2':              r2,
    'Gamma_vec':       Gamma_vec.tolist(),
    'n_obs':           int(n),
    'sigma_price':     sigma_price,
    'p_price':         p_p,
    'lag0_ccf':        lag0_corr,
    'note': (
        'gamma0 is the OLS coupling strength dL^X = gamma0*dL^Y + eps. '
        'Gamma_vec = gamma0 * B_X (state-space coupling vector). '
        'Interpretation: price CARMA driven by dL^X + gamma0*dL^Y.'
    ),
}

out_path = RES_DIR / 'coupling_params.json'
with open(out_path, 'w') as f:
    json.dump(coupling_params, f, indent=2)
print(f'Saved → {out_path}')
print()
print('Summary:')
print(f'  γ₀ = {gamma0:.6f}  (p={p_val:.2e})')
print(f'  Γ  = {np.round(Gamma_vec, 6)}')